**1. Install and Import Libraries**

In [8]:
!pip install opencv-python mediapipe scikit-learn matplotlib tensorflow


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import cv2
import numpy as np
import os
import time
from matplotlib import pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print(os.listdir())

['.ipynb_checkpoints', '0.npy', 'action_model.keras', 'asl_action_model.h5', 'asl_action_model.keras', 'ASL_Data', 'CodeFile.ipynb', 'hand_landmarker.task', 'Logs', 'pose_landmarker.task', 'Untitled.ipynb', 'untitled.py', 'Untitled1.ipynb', 'Untitled2.ipynb', 'Untitled3.ipynb']


**Initialize MediaPipe Hand and Pose Landmark Detectors**

In [10]:
# Hand detector
hand_base = python.BaseOptions(model_asset_path="hand_landmarker.task")

hand_options = vision.HandLandmarkerOptions(
    base_options=hand_base,
    num_hands=2
)

hand_detector = vision.HandLandmarker.create_from_options(hand_options)


# Pose detector
pose_base = python.BaseOptions(model_asset_path="pose_landmarker.task")

pose_options = vision.PoseLandmarkerOptions(
    base_options=pose_base
)

pose_detector = vision.PoseLandmarker.create_from_options(pose_options)

In [11]:
def mediapipe_detection(frame):

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    hand_results = hand_detector.detect(mp_image)
    pose_results = pose_detector.detect(mp_image)

    return hand_results, pose_results

In [12]:
def draw_styled_landmarks(frame, hand_results, pose_results):

    # HAND CONNECTIONS
    HAND_CONNECTIONS = [
        (0,1),(1,2),(2,3),(3,4),
        (0,5),(5,6),(6,7),(7,8),
        (5,9),(9,10),(10,11),(11,12),
        (9,13),(13,14),(14,15),(15,16),
        (13,17),(17,18),(18,19),(19,20),
        (0,17)
    ]

    if hand_results.hand_landmarks:
        for hand in hand_results.hand_landmarks:

            # draw points
            for lm in hand:
                x = int(lm.x * frame.shape[1])
                y = int(lm.y * frame.shape[0])
                cv2.circle(frame, (x,y), 4, (0,255,0), -1)

            # draw lines
            for connection in HAND_CONNECTIONS:
                start = hand[connection[0]]
                end = hand[connection[1]]

                x1 = int(start.x * frame.shape[1])
                y1 = int(start.y * frame.shape[0])
                x2 = int(end.x * frame.shape[1])
                y2 = int(end.y * frame.shape[0])

                cv2.line(frame, (x1,y1), (x2,y2), (0,255,255), 2)


    # POSE CONNECTIONS
    POSE_CONNECTIONS = [
        (11,13),(13,15),
        (12,14),(14,16),
        (11,12)
    ]

    if pose_results.pose_landmarks:

        pose = pose_results.pose_landmarks[0]

        for lm in pose:
            x = int(lm.x * frame.shape[1])
            y = int(lm.y * frame.shape[0])
            cv2.circle(frame, (x,y), 3, (255,0,0), -1)

        for connection in POSE_CONNECTIONS:
            start = pose[connection[0]]
            end = pose[connection[1]]

            x1 = int(start.x * frame.shape[1])
            y1 = int(start.y * frame.shape[0])
            x2 = int(end.x * frame.shape[1])
            y2 = int(end.y * frame.shape[0])

            cv2.line(frame, (x1,y1), (x2,y2), (255,255,0), 2)

In [13]:
cap = cv2.VideoCapture(0)

In [14]:
while cap.isOpened():

    ret, frame = cap.read()

    if not ret:
        break

    hand_results, pose_results = mediapipe_detection(frame)

    draw_styled_landmarks(frame, hand_results, pose_results)

    cv2.imshow("ASL Detection Feed", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

**Extract Landmark Feature Vectors for Model Training**

In [15]:
def extract_pose(pose_results):

    if pose_results.pose_landmarks:

        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility]
             for lm in pose_results.pose_landmarks[0]]
        ).flatten()

    else:
        pose = np.zeros(33*4)

    return pose

In [16]:
def extract_hands(hand_results):

    left = np.zeros(21*3)
    right = np.zeros(21*3)

    if hand_results.hand_landmarks:

        for idx, hand in enumerate(hand_results.hand_landmarks):

            hand_array = np.array(
                [[lm.x, lm.y, lm.z] for lm in hand]
            ).flatten()

            if idx == 0:
                left = hand_array
            elif idx == 1:
                right = hand_array

    return left, right

In [17]:
def extract_keypoints(hand_results, pose_results):

    pose = extract_pose(pose_results)

    left, right = extract_hands(hand_results)

    return np.concatenate([pose, left, right])

In [18]:
keypoints = extract_keypoints(hand_results, pose_results)

print("Feature vector length:", len(keypoints))

Feature vector length: 258


In [19]:
result_test = extract_keypoints(hand_results, pose_results)

In [20]:
33*4 + 21*3 + 21*3

258

In [21]:
np.save('0', result_test)
np.load('0.npy')

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

**Create Dataset Directory Structure for Training Data**

In [22]:
import os

In [23]:
DATA_PATH = os.path.join("ASL_Data")

In [24]:
no_sequences = 30
sequence_length = 30

In [25]:
actions = np.array([
    "hello",
    "thanks",
    "iloveyou",
    "yes",
    "no"
])

In [26]:
for action in actions:
    for sequence in range(no_sequences):
        dir_path = os.path.join(DATA_PATH, action, str(sequence))
        os.makedirs(dir_path, exist_ok=True)

**Collect Training Sequences and Save Landmark Keypoints**

In [111]:
cap = cv2.VideoCapture(0)

In [27]:
stop_collection = False 

In [158]:
for action in actions:
    for sequence in range(no_sequences):
        for frame_num in range(sequence_length):

            ret, frame = cap.read()
            if not ret:
                stop_collection = True
                break

            hand_results, pose_results = mediapipe_detection(frame) # Run MediaPipe detection
            draw_styled_landmarks(frame, hand_results, pose_results) # Draw landmarks
            keypoints = extract_keypoints(hand_results, pose_results) # Extract keypoints

            npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num)) # Save keypoints
            np.save(npy_path, keypoints)

            # Display status on screen
            cv2.putText(
                frame,
                f"Collecting {action} | Video {sequence}",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            cv2.imshow("ASL Data Collection", frame)

            if cv2.waitKey(10) & 0xFF == ord('q'):
                stop_collection = True
                break

        if stop_collection:
            break
    if stop_collection:
        break

In [159]:
cap.release()
cv2.destroyAllWindows()

**Dataset Assembly & Train/Test Split**

In [28]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

In [29]:
label_map = {label: num for num, label in enumerate(actions)}
label_map

{np.str_('hello'): 0,
 np.str_('thanks'): 1,
 np.str_('iloveyou'): 2,
 np.str_('yes'): 3,
 np.str_('no'): 4}

In [30]:
sequences = []
labels = []

for action in actions:
    for sequence in range(no_sequences):

        window = []
        valid_sequence = True

        for frame_num in range(sequence_length):

            file_path = os.path.join(
                DATA_PATH, action, str(sequence), f"{frame_num}.npy"
            )

            if not os.path.exists(file_path):
                valid_sequence = False
                break

            res = np.load(file_path)
            window.append(res)

        if valid_sequence:
            sequences.append(window)
            labels.append(label_map[action])

In [31]:
X = np.array(sequences)
y = to_categorical(labels).astype(int)

print("X shape:",X.shape)
print("y shape:",y.shape)

X shape: (150, 30, 258)
y shape: (150, 5)


In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.05,
    shuffle=True,
    random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (142, 30, 258)
Test: (8, 30, 258)


In [33]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(142, 30, 258)
(142, 5)
(8, 30, 258)
(8, 5)


**Build and train LSTM Neural Network**

In [34]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.callbacks import TensorBoard
import os

In [35]:
log_dir = os.path.join("Logs")
tb_callback = TensorBoard(log_dir=log_dir)

In [36]:
model = Sequential()
model.add(Input(shape=(30, X.shape[2])))

model.add(LSTM(64, return_sequences=True, activation='relu'))
model.add(LSTM(128, return_sequences=True, activation='relu'))
model.add(LSTM(64, activation='relu'))

model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))

model.add(Dense(actions.shape[0], activation='softmax'))

In [37]:
model.compile(
    optimizer='Adam',
    loss='categorical_crossentropy',
    metrics=['categorical_accuracy']
)

In [38]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 30, 64)              │          82,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 30, 128)             │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ (None, 64)                  │          49,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 5)                   │             165 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 237,317 (927.02 KB)

 Trainable params: 237,317 (927.02 KB)

 Non-trainable params: 0 (0.00 B)

In [39]:
pip install tensorboard

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: C:\Users\nithy\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [40]:
model.fit(
    X_train,
    y_train,
    epochs=200,
    validation_data=(X_test, y_test),
    callbacks=[tb_callback]
)

Epoch 1/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 10s 351ms/step - categorical_accuracy: 0.1761 - loss: 1.6755 - val_categorical_accuracy: 0.1250 - val_loss: 1.6141
Epoch 2/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 104ms/step - categorical_accuracy: 0.2113 - loss: 1.5887 - val_categorical_accuracy: 0.1250 - val_loss: 1.9889
Epoch 3/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/step - categorical_accuracy: 0.2606 - loss: 1.5979 - val_categorical_accuracy: 0.1250 - val_loss: 1.6304
Epoch 4/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 96ms/step - categorical_accuracy: 0.2817 - loss: 1.5568 - val_categorical_accuracy: 0.1250 - val_loss: 1.9710
Epoch 5/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 100ms/step - categorical_accuracy: 0.3028 - loss: 1.4821 - val_categorical_accuracy: 0.1250 - val_loss: 1.7775
Epoch 6/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - categorical_accuracy: 0.3239 - loss: 1.4630 - val_categorical_accuracy: 0.1250 - val_loss: 1.6417
Epoch 7/200
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - categorical_accuracy: 0.4225 - loss: 1.46

In [41]:
model.save("asl_action_model.keras")

**Make Predictions**

In [42]:
res = model.predict(X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 740ms/step


In [43]:
actions[np.argmax(res[4])]

np.str_('no')

In [44]:
actions[np.argmax(y_test[4])]

np.str_('iloveyou')

**Save Weights**

In [45]:
model.save("asl_action_model.keras")

In [46]:
from tensorflow.keras.models import load_model

In [47]:
model = load_model("asl_action_model.keras")

**Evaluation using Confusion Matrix and Accuracy**

In [48]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [49]:
y_pred = model.predict(X_train)

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 200ms/step


In [50]:
y_true = np.argmax(y_train, axis=1)
y_pred = np.argmax(y_pred, axis=1)

In [51]:
multilabel_confusion_matrix(y_true, y_pred)

array([[[109,   4],
        [  7,  22]],

       [[112,   1],
        [ 28,   1]],

       [[116,   0],
        [ 26,   0]],

       [[113,   0],
        [ 21,   8]],

       [[ 36,  77],
        [  0,  29]]])

In [52]:
y_pred = model.predict(X_test)

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(y_pred, axis=1)

multilabel_confusion_matrix(y_true, y_pred)
accuracy_score(y_true, y_pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step


0.25

**Real_Time Testing**

In [96]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

In [97]:
model = load_model("asl_action_model.keras")

In [98]:
cap = cv2.VideoCapture(0)

In [99]:
sequence = []
sentence = []
predictions=[]
threshold = 0.8

In [100]:
while True:

    ret, frame = cap.read()
    if not ret:
        print("Frame not captured")
        break

    # MediaPipe detection
    hand_results, pose_results = mediapipe_detection(frame)

    # Draw landmarks
    draw_styled_landmarks(frame, hand_results, pose_results)

    # Extract keypoints
    keypoints = extract_keypoints(hand_results, pose_results)

    sequence.append(keypoints)
    sequence = sequence[-30:]

    # Prediction
    if len(sequence) == 30:

        res = model.predict(np.expand_dims(sequence, axis=0))[0]

        predictions.append(np.argmax(res))
        predictions = predictions[-10:]

        if np.unique(predictions)[0] == np.argmax(res):

            if res[np.argmax(res)] > threshold:

                if len(sentence) == 0 or actions[np.argmax(res)] != sentence[-1]:
                    sentence.append(actions[np.argmax(res)])

        if len(sentence) > 5:
            sentence = sentence[-5:]

    # Display prediction
    cv2.rectangle(frame, (0,0), (640,40), (245,117,16), -1)

    cv2.putText(
        frame,
        ' '.join(sentence),
        (10,30),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255,255,255),
        2
    )

    cv2.imshow("ASL Recognition", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 411ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━

In [101]:
cap.release()
cv2.destroyAllWindows()

In [102]:
res = model.predict(X_test)

for i in range(10):
    print("Predicted:", actions[np.argmax(res[i])],
          " | True:", actions[np.argmax(y_test[i])])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 812ms/step
Predicted: no  | True: iloveyou
Predicted: hello  | True: hello
Predicted: no  | True: yes
Predicted: no  | True: iloveyou
Predicted: no  | True: iloveyou
Predicted: no  | True: thanks
Predicted: no  | True: iloveyou
Predicted: no  | True: no


IndexError: index 8 is out of bounds for axis 0 with size 8